# Insert NKG → Neo4j

Reads all `.nkg.json` files from `data/nkg_gpu3_fix_orphans` and upserts them into Neo4j.

**Key design decision — `nkg_id`:**  
Elements from different pages can share the same DOM `id` (e.g. `#content`).  
To keep them as distinct nodes, every Element gets a scoped key:  
`nkg_id = "{page_id}/{element_id}"` e.g. `/customer/topup/content`  
The original `id` and `selector` are stored as plain properties for the frontend to use.

In [7]:
import os
import json
from glob import glob
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
print("Env loaded.")

Env loaded.


In [8]:
NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j:", NEO4J_URI)

Connected to Neo4j: bolt://localhost:7687


## 1 — Create constraints (run once)
Uniqueness constraints automatically create an index, making every `MERGE` O(log n) instead of a full scan.

In [9]:
CONSTRAINTS = [
    "CREATE CONSTRAINT page_id IF NOT EXISTS FOR (p:Page) REQUIRE p.id IS UNIQUE",
    "CREATE CONSTRAINT element_nkg_id IF NOT EXISTS FOR (e:Element) REQUIRE e.nkg_id IS UNIQUE",
]

with driver.session() as s:
    for q in CONSTRAINTS:
        s.run(q)
print("Constraints ready.")

Constraints ready.


## 2 — Cypher queries
Each query handles exactly ONE concern — no chained `UNWIND`s that cause Cartesian products.

**Critical fix for triggers:** Element-type triggers use `MATCH` (not `MERGE`) so Neo4j will
simply skip a trigger if the target doesn't already exist — preventing ghost/orphan node creation.

In [10]:
# Upsert the Page node
Q_PAGE = """
MERGE (p:Page {id: $page_id})
SET p.title = $title, p.desc = $desc
"""

# Upsert Element nodes (nkg_id is the unique key)
Q_ELEMENTS = """
UNWIND $elements AS e
MERGE (el:Element {nkg_id: e.nkg_id})
SET el.id       = e.id,
    el.page_id  = e.page_id,
    el.desc     = e.desc,
    el.type     = e.type,
    el.selector = e.selector,
    el.text     = e.text
"""

# Page -[:CONTAINS]-> Element
Q_PAGE_CONTAINS = """
UNWIND $contains AS c
MATCH (p:Page    {id:     c.page_id})
MATCH (el:Element{nkg_id: c.nkg_id})
MERGE (p)-[:CONTAINS]->(el)
"""

# Element -[:CONTAINS]-> Element  (nested UI, e.g. modal > button)
Q_ELEMENT_CONTAINS = """
UNWIND $element_contains AS c
MATCH (parent:Element {nkg_id: c.parent_nkg_id})
MATCH (child :Element {nkg_id: c.child_nkg_id})
MERGE (parent)-[:CONTAINS]->(child)
"""

# Element -[:TRIGGERS]-> Element
# Uses MATCH (not MERGE) — if target element doesn't exist, the row is silently skipped.
# This prevents ghost/orphan nodes from hallucinated or schema-placeholder trigger targets.
Q_TRIGGERS_ELEMENT = """
UNWIND $triggers AS t
MATCH (from:Element {nkg_id: t.from_nkg_id})
MATCH (toEl:Element {nkg_id: t.to_nkg_id})
MERGE (from)-[:TRIGGERS]->(toEl)
"""

# Element -[:TRIGGERS]-> Page  (MERGE is intentional: page nodes are always valid targets)
Q_TRIGGERS_PAGE = """
UNWIND $triggers AS t
MATCH (from:Element {nkg_id: t.from_nkg_id})
MERGE (toPg:Page {id: t.to})
MERGE (from)-[:TRIGGERS]->(toPg)
"""

print("Queries defined.")

Queries defined.


## 3 — Param enrichment
Adds `nkg_id` to every element/relationship **before** sending to Neo4j.

The JSON source files are pre-cleaned by `fix_orphans.py`, so `enrich()` simply
translates raw IDs into scoped `nkg_id`s. The MATCH-based Cypher queries act as
a last-resort safety net in case any bad data slips through.

In [11]:
def enrich(raw_params: dict) -> dict:
    """
    Transform NKG data (or legacy cypher_payload.params) into Neo4j-ready dicts.
    Adds nkg_id = "{page_id}/{element_id}" to elements and all relationship refs.
    Filters bad trigger targets to prevent ghost node creation.

    The NKG JSON format does not have an explicit 'contains' array — every element
    in the list IS contained by the page, so we derive it from the elements list.
    """
    page_id = raw_params["page"]["id"]

    def nkg(elem_id: str) -> str:
        return f"{page_id}/{elem_id}"

    # Elements
    elements = [
        {
            "nkg_id":   nkg(e["id"]),
            "id":       e["id"],
            "page_id":  page_id,
            "desc":     e.get("desc", ""),
            "type":     e.get("type", ""),
            "selector": e.get("selector", ""),
            "text":     e.get("text", ""),
        }
        for e in raw_params.get("elements", [])
    ]

    # Build a set of valid element IDs for this page — used to filter triggers
    valid_elem_ids = {e["id"] for e in raw_params.get("elements", []) if e.get("id")}

    # Page -> Element CONTAINS
    # Old cypher_payload.params had an explicit 'contains' list.
    # The new NKG format doesn't — every element in the list belongs to the page.
    raw_contains = raw_params.get("contains", [])
    if raw_contains:
        # Legacy path: explicit contains list with element_id keys
        contains = [
            {"page_id": page_id, "nkg_id": nkg(c["element_id"])}
            for c in raw_contains
        ]
    else:
        # NKG path: derive contains from all elements
        contains = [
            {"page_id": page_id, "nkg_id": nkg(e["id"])}
            for e in raw_params.get("elements", [])
        ]

    # Element -> Element CONTAINS
    element_contains = [
        {
            "parent_nkg_id": nkg(c["parent_id"]),
            "child_nkg_id":  nkg(c["child_id"]),
        }
        for c in raw_params.get("element_contains", [])
    ]

    # TRIGGERS — split into element-type and page-type
    elem_triggers = []
    page_triggers = []
    skipped_triggers = 0

    for t in raw_params.get("triggers", []):
        from_id  = t.get("from", "")
        to_id    = t.get("to", "")
        to_type  = t.get("to_type", "")

        if not from_id or not to_id:
            skipped_triggers += 1
            continue

        if to_type == "element":
            if to_id not in valid_elem_ids:
                skipped_triggers += 1
                continue
            elem_triggers.append({
                "from_nkg_id": nkg(from_id),
                "to_nkg_id":   nkg(to_id),
            })

        elif to_type == "page":
            page_triggers.append({
                "from_nkg_id": nkg(from_id),
                "to":          to_id,
            })

        else:
            skipped_triggers += 1

    return {
        "page_id":           page_id,
        "title":             raw_params["page"].get("title", ""),
        "desc":              raw_params["page"].get("desc", ""),
        "elements":          elements,
        "contains":          contains,
        "element_contains":  element_contains,
        "elem_triggers":     elem_triggers,
        "page_triggers":     page_triggers,
        "skipped_triggers":  skipped_triggers,
    }

print("enrich() ready.")

enrich() ready.


## 4 — Insert all files

In [12]:
data_dir   = "../data/nkg_gpu3_fix_orphans_with_text"
json_files = sorted(glob(os.path.join(data_dir, "*.json")))
print(f"Found {len(json_files)} NKG JSON files.\n")

ok, skipped, errors = 0, 0, []
total_skipped_triggers = 0

with driver.session() as session:
    for filepath in json_files:
        fname = os.path.basename(filepath)
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Use 'nkg' first, fallback to 'cypher_payload'
            params = data.get("nkg") or data.get("cypher_payload", {}).get("params")

            if not params:
                print(f"  [SKIP] {fname} — no usable data found")
                skipped += 1
                continue

            p = enrich(params)
            total_skipped_triggers += p["skipped_triggers"]

            session.run(Q_PAGE,     page_id=p["page_id"], title=p["title"], desc=p["desc"])
            session.run(Q_ELEMENTS, elements=p["elements"])
            session.run(Q_PAGE_CONTAINS, contains=p["contains"])

            if p["element_contains"]:
                session.run(Q_ELEMENT_CONTAINS, element_contains=p["element_contains"])

            if p["elem_triggers"]:
                session.run(Q_TRIGGERS_ELEMENT, triggers=p["elem_triggers"])

            if p["page_triggers"]:
                session.run(Q_TRIGGERS_PAGE, triggers=p["page_triggers"])

            skip_note = f"  [{p['skipped_triggers']} bad triggers dropped]" if p["skipped_triggers"] else ""
            print(f"  [OK] {fname}  ({len(p['elements'])} elements, "
                  f"{len(p['elem_triggers'])} elem-triggers, "
                  f"{len(p['page_triggers'])} page-triggers){skip_note}")
            ok += 1

        except Exception as e:
            print(f"  [ERR] {fname}: {e}")
            errors.append((fname, str(e)))

print(f"\nDone. ok={ok}  skipped={skipped}  errors={len(errors)}")
print(f"Total bad triggers dropped (ghost prevention): {total_skipped_triggers}")
if errors:
    for name, msg in errors:
        print(f"  ✗ {name}: {msg}")

Found 47 NKG JSON files.

  [OK] customer_attendance_day_off.nkg.json  (63 elements, 7 elem-triggers, 1 page-triggers)
  [OK] customer_attendance_employee_schedule.nkg.json  (167 elements, 2 elem-triggers, 0 page-triggers)
  [OK] customer_attendance_leave.nkg.json  (332 elements, 58 elem-triggers, 3 page-triggers)
  [OK] customer_attendance_leave_allowance.nkg.json  (47 elements, 11 elem-triggers, 0 page-triggers)
  [OK] customer_attendance_overtime.nkg.json  (130 elements, 15 elem-triggers, 4 page-triggers)
  [OK] customer_attendance_schedule.nkg.json  (59 elements, 6 elem-triggers, 0 page-triggers)
  [OK] customer_attendance_status.nkg.json  (124 elements, 22 elem-triggers, 0 page-triggers)
  [OK] customer_cart.nkg.json  (512 elements, 85 elem-triggers, 3 page-triggers)
  [OK] customer_changelog.nkg.json  (39 elements, 0 elem-triggers, 2 page-triggers)
  [OK] customer_dashboard.nkg.json  (78 elements, 10 elem-triggers, 5 page-triggers)
  [OK] customer_device.nkg.json  (100 elements, 

## 5 — Quick verification

In [13]:
with driver.session() as session:
    result = session.run("""
        MATCH (p:Page)
        OPTIONAL MATCH (p)-[:CONTAINS]->(e:Element)
        RETURN p.id AS page, count(e) AS elements
        ORDER BY page
    """)
    rows = result.data()

total_elements = sum(r["elements"] for r in rows)
print(f"{len(rows)} pages | {total_elements} page→element edges\n")
for r in rows:
    print(f"  {r['page']:<45} {r['elements']:>4} elements")

60 pages | 4365 page→element edges

  /customer                                        0 elements
  /customer/attendance/day_off                    63 elements
  /customer/attendance/employee_schedule         167 elements
  /customer/attendance/leave                     332 elements
  /customer/attendance/leave_allowance            47 elements
  /customer/attendance/overtime                  130 elements
  /customer/attendance/overtime/store              0 elements
  /customer/attendance/overtime/update             0 elements
  /customer/attendance/schedule                   59 elements
  /customer/attendance/status                    124 elements
  /customer/cart                                   0 elements
  /customer/changelog                             39 elements
  /customer/dashboard                             78 elements
  /customer/device                               100 elements
  /customer/device/add                             0 elements
  /customer/device/api_sdk        

In [14]:
# Orphan check — should return 0 after the fix
with driver.session() as session:
    result = session.run("""
        MATCH (e:Element)
        WHERE NOT ()-[:CONTAINS]->(e)
        RETURN count(e) AS orphans
    """)
    count = result.single()["orphans"]

if count == 0:
    print("✓ No orphans — graph is clean.")
else:
    print(f"✗ {count} orphan(s) still found. Run the orphan query below to inspect.")

✓ No orphans — graph is clean.


## 6 — Create vector index

A **HNSW vector index** enables fast approximate nearest-neighbour search on the `embedding`
property. Without this index, every similarity query would require a full scan of all Element
nodes — O(n) per query. With the index it is O(log n).

Key options:
- `vector.dimensions` — must match the output size of your embedding model exactly.
  `qwen3-embedding` (0.6b) → **1024**, (4b) → **2560**, (8b) → **4096**. Change if needed.
- `vector.similarity_function` — `cosine` is standard for instruction-tuned embedding models.

> **Run this cell once**, before or after inserting embeddings. It is idempotent (`IF NOT EXISTS`).

In [15]:
# ── Change this to match your qwen3-embedding variant ──────────────────────
EMBEDDING_DIMENSIONS = 1024   # qwen3-embedding:0.6b=1024 | 4b=2560 | 8b=4096
# ───────────────────────────────────────────────────────────────────────────

Q_VECTOR_INDEX = f"""
CREATE VECTOR INDEX element_embedding_idx IF NOT EXISTS
FOR (e:Element)
ON e.embedding
OPTIONS {{
    indexConfig: {{
        `vector.dimensions`:         {EMBEDDING_DIMENSIONS},
        `vector.similarity_function`: 'cosine'
    }}
}}
"""

with driver.session() as session:
    session.run(Q_VECTOR_INDEX)

    # Wait for index to come ONLINE (it builds asynchronously)
    import time
    for attempt in range(30):
        result = session.run("""
            SHOW INDEXES YIELD name, state
            WHERE name = 'element_embedding_idx'
            RETURN state
        """)
        row = result.single()
        state = row["state"] if row else "NOT FOUND"
        if state == "ONLINE":
            break
        print(f"  Index state: {state} — waiting...")
        time.sleep(2)

print(f"✓ Vector index 'element_embedding_idx' is {state}")
print(f"  dimensions={EMBEDDING_DIMENSIONS}, similarity=cosine")

✓ Vector index 'element_embedding_idx' is ONLINE
  dimensions=1024, similarity=cosine


## 7 — Load & insert embeddings

Reads `embeddings_neo4j.jsonl` (output of `embed/embed.py`) and sets `el.embedding`
on every matching Element node.

**Performance strategy — batched UNWIND:**  
Sending one record per transaction = one round-trip per element (~4 500 transactions).  
Instead, we chunk the JSONL into batches of `BATCH_SIZE` and send each batch as one
transaction using `UNWIND`. This reduces round-trips by ~100x.

**`db.create.setNodeVectorProperty` vs plain `SET`:**  
Neo4j ≥ 5.11 requires using `db.create.setNodeVectorProperty` (or its alias
`CALL db.create.setNodeVectorProperty`) to store a list as a proper vector type that
the HNSW index can work with. A plain `SET e.embedding = row.embedding` stores a
generic list, which the vector index cannot read.

> **If you are on Neo4j 5.18+** you can also use `SET e.embedding = row.embedding`
> directly — Neo4j will coerce it automatically when an index exists.

In [16]:
import time

EMBEDDINGS_JSONL = "./neo4j-query/embeddings_neo4j_with_text.jsonl"
BATCH_SIZE       = 500   # records per transaction — tune up/down based on RAM

# Cypher: UNWIND a batch, MATCH each Element, set vector property
Q_SET_EMBEDDING = """
UNWIND $batch AS row
MATCH (e:Element {nkg_id: row.nkg_id})
CALL db.create.setNodeVectorProperty(e, 'embedding', row.embedding)
"""

# ── Load the JSONL ──────────────────────────────────────────────────────────
records = []
with open(EMBEDDINGS_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} embedding records from {EMBEDDINGS_JSONL}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Batches    : {-(-len(records) // BATCH_SIZE)}  (ceil division)")
print("-" * 50)

# ── Batch insert ─────────────────────────────────────────────────────────────
ok_count  = 0
err_count = 0
t_start   = time.time()

with driver.session() as session:
    for batch_start in range(0, len(records), BATCH_SIZE):
        batch = records[batch_start : batch_start + BATCH_SIZE]
        batch_end = batch_start + len(batch)
        try:
            session.run(Q_SET_EMBEDDING, batch=batch)
            ok_count += len(batch)
            elapsed = time.time() - t_start
            rate    = ok_count / elapsed
            eta     = (len(records) - ok_count) / rate if rate > 0 else 0
            print(f"  [{batch_end:>5}/{len(records)}]  "
                  f"batch {batch_start//BATCH_SIZE + 1}  "
                  f"({len(batch)} records)  ETA={eta:.0f}s")
        except Exception as exc:
            print(f"  [ERR] batch {batch_start}–{batch_end}: {exc}")
            err_count += len(batch)

elapsed = time.time() - t_start
print(f"\nDone in {elapsed:.1f}s — set={ok_count}  errors={err_count}")

Loaded 4365 embedding records from ./neo4j-query/embeddings_neo4j_with_text.jsonl
Batch size : 500
Batches    : 9  (ceil division)
--------------------------------------------------
  [  500/4365]  batch 1  (500 records)  ETA=53s
  [ 1000/4365]  batch 2  (500 records)  ETA=53s
  [ 1500/4365]  batch 3  (500 records)  ETA=48s
  [ 2000/4365]  batch 4  (500 records)  ETA=40s
  [ 2500/4365]  batch 5  (500 records)  ETA=32s
  [ 3000/4365]  batch 6  (500 records)  ETA=24s
  [ 3500/4365]  batch 7  (500 records)  ETA=15s
  [ 4000/4365]  batch 8  (500 records)  ETA=6s
  [ 4365/4365]  batch 9  (365 records)  ETA=0s

Done in 79.7s — set=4365  errors=0


## 8 — Verify embedding coverage

In [17]:
with driver.session() as session:
    result = session.run("""
        MATCH (e:Element)
        RETURN
            count(e)                                             AS total,
            count(e.embedding)                                   AS with_embedding,
            count(e) - count(e.embedding)                        AS without_embedding
    """)
    row = result.single()

total   = row["total"]
with_e  = row["with_embedding"]
without = row["without_embedding"]
pct     = (with_e / total * 100) if total else 0

print(f"Total elements      : {total}")
print(f"With embedding      : {with_e}  ({pct:.1f}%)")
print(f"Without embedding   : {without}")

if without == 0:
    print("\n✓ All elements have embeddings — vector index is fully populated.")
else:
    print(f"\n⚠ {without} element(s) have no embedding.")
    print("  These are elements whose nkg_id was not found in the JSONL.")
    print("  Re-run embed/embed.py with --resume, then re-run cell 7.")

Total elements      : 4365
With embedding      : 4365  (100.0%)
Without embedding   : 0

✓ All elements have embeddings — vector index is fully populated.


In [ ]:
driver.close()
print("Connection closed.")